# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and processing the FAIR² protein dataset using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and explore its contents using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print main metadata description
print("Name:", dataset.metadata.name)
print("Identifier:", getattr(dataset.metadata, 'identifier', 'N/A'))
print("Description:", dataset.metadata.description)
print("Published:", getattr(dataset.metadata, 'datePublished', 'N/A'))
print("License:", dataset.metadata.license)
print("Keywords:", getattr(dataset.metadata, 'keywords', []))

## 2. Data Overview

Examine available record sets (tables), field definitions, and their unique Croissant `@id` identifiers.

In [ ]:
# List all RecordSet (@id, name, description) for the dataset
print("Available record sets in the dataset:\n")

record_sets = dataset.metadata.record_sets
for r in record_sets:
    print(f"@id: {r.id}")
    print(f"  name: {getattr(r, 'name', 'N/A')}")
    print(f"  description: {getattr(r, 'description', 'N/A')}")
    print("")

# For each RecordSet, show its fields and field @id
print("Fields in each record set:")
for r in record_sets:
    print(f"RecordSet @id: {r.id}  ({getattr(r, 'name', '-')})")
    if getattr(r, 'fields', None):
        for field in r.fields:
            print(f"    Field @id: {field.id:<40} name: {getattr(field, 'name', '-')}")
    else:
        print("    No fields defined.")
    print("")

## 3. Data Extraction

Load records from a specific record set into a DataFrame using its `@id` and show fields (columns) for analysis.

In [ ]:
# List all record set @id for extraction
all_record_set_ids = [r.id for r in dataset.metadata.record_sets]
print("Extracting data for record sets:", all_record_set_ids)

dataframes = {}
for record_set_id in all_record_set_ids:
    # Extract records using Croissant @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nRecordSet {record_set_id} DataFrame shape: {df.shape}")
    print(df.columns.tolist())

# Just pick the first record set to preview its data
selected_record_set_id = all_record_set_ids[0]
print(f"\nPreview of data for record set {selected_record_set_id}:")
dataframes[selected_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Demonstrate common data processing: filtering, normalization, and grouping. This section uses `@id` references for fields.

In [ ]:
# Pick a numeric field and group field using their @id (update ids as in record set field listing)
# For demonstration, select first record set and check its numeric fields
df = dataframes[selected_record_set_id]

print(f"\nColumns in RecordSet (@id: {selected_record_set_id}): \n{df.columns.tolist()}")

# Try to guess a numeric field by checking types or common field names
numeric_candidates = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] or 'count' in col.lower() or 'abundance' in col.lower()]

if not numeric_candidates:
    # Try to coerce one if all columns are objects (likely numeric as string)
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            if df[col].dtype in ['float64', 'int64']:
                numeric_candidates.append(col)
        except Exception:
            continue

print("Numeric candidates: ", numeric_candidates)

# Select field for demonstration
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    numeric_field_id = df.columns[0]  # fallback

print(f"Using numeric field: {numeric_field_id}\n")

# Example threshold
threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype in ['float64', 'int64'] else 0

filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a non-numeric field
group_candidates = [f for f in df.columns if f != numeric_field_id and df[f].dtype == 'object']
if group_candidates:
    group_field_id = group_candidates[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization

Visualize distribution of a numeric field and relationships using Matplotlib/Seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(data=df, x=numeric_field_id, bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# If a suitable group field exists, boxplot
if group_candidates:
    plt.figure(figsize=(10,4))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

This notebook demonstrated end-to-end loading and exploration of the FAIR² mass spectrometry protein dataset using the `mlcroissant` library.

**Key steps covered:**
- Loading dataset metadata and understanding its Croissant structure
- Inspecting record sets and their field `@id`s
- Extracting and processing records by entity identifier
- Basic exploratory analysis: filtering, normalization, grouping
- Visualization of value distributions and relationships

This FAIR² dataset offers rich protein information suitable for downstream statistical and biomarker discovery analyses.